# Running a Finance Company with CrewAI Flows and Parallel Agents


### Step 1: Setup


In [ ]:
%pip install -q "crewai[tools]>=1.6.1,<2" python-dotenv pydantic exa-py

import os

from crewai.llm import LLM
from crewai_tools import EXASearchTool
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
search_tool = EXASearchTool()

### Step 2: Event-Driven Flows


In [ ]:
from crewai.flow.flow import Flow, start, listen


class TinyCompanyFlow(Flow):
    @start()
    def open_office(self):
        print("Office opened")
        return "market opens"

    @listen(open_office)
    def morning_meeting(self, opening_message):
        print(f"Morning meeting starts because: {opening_message}")
        return "meeting complete"


tiny_flow = TinyCompanyFlow()
_ = await tiny_flow.kickoff_async()

### Step 3: Flow State


In [ ]:
from pydantic import BaseModel, Field


class FinanceCompanyState(BaseModel):
    company: str = Field(default="NVIDIA", description="Company being evaluated")
    market_research: str = ""
    risk_assessment: str = ""
    risk_level: str = "normal"
    compliance_notes: str = ""
    portfolio_strategy: str = ""
    client_report: str = ""


class StateDemoFlow(Flow[FinanceCompanyState]):
    @start()
    def inspect_state(self):
        print(f"Company under review: {self.state.company}")
        self.state.market_research = "Market research will be stored here."
        print(self.state.market_research)


state_demo = StateDemoFlow()
_ = await state_demo.kickoff_async()

### Step 4: Crew Inside a Flow Step


In [ ]:
from datetime import datetime

from crewai import Agent, Crew, Process, Task


class MarketResearchFlow(Flow[FinanceCompanyState]):
    @start()
    def run_market_research(self):
        researcher = Agent(
            role="Market Researcher",
            goal="Research the latest market position and investor narrative for {company}",
            backstory="You are a finance research analyst who finds concise, source-grounded market context.",
            llm=llm,
            tools=[search_tool],
            verbose=False,
        )

        task = Task(
            description=(
                "Research {company} as of {date}. "
                "Find 3 important market developments, summarize each one, and explain why investors care."
            ),
            expected_output="A concise markdown brief with: 3 market developments, investor relevance, and source names or URLs.",
            agent=researcher,
            output_file="task_outputs/step_4/step4_market_research.md",
        )

        crew = Crew(
            agents=[researcher], 
            tasks=[task], 
            process=Process.sequential, 
            verbose=True
            )

        result = crew.kickoff(
            inputs={"company": self.state.company, "date": datetime.now().strftime("%B %d, %Y")})
        self.state.market_research = str(result)
        return self.state.market_research


market_flow = MarketResearchFlow()
market_flow.state.company = "NVIDIA"
_ = await market_flow.kickoff_async()

### Step 5: Sequential `@listen`


In [ ]:
class SequentialFinanceFlow(Flow[FinanceCompanyState]):
    @start()
    def run_market_research(self):
        researcher = Agent(
            role="Market Researcher",
            goal="Research recent market developments for {company}",
            backstory="You find concise, source-grounded finance updates for investors.",
            llm=llm,
            tools=[search_tool],
            verbose=False,
        )

        task = Task(
            description="Find 3 recent market developments about {company} as of {date} and explain why investors care.",
            expected_output="A concise markdown market brief with sources.",
            agent=researcher,
            output_file="task_outputs/step_5/step5_market_research.md",
        )

        crew = Crew(agents=[researcher], tasks=[task], process=Process.sequential, verbose=True)
        result = crew.kickoff(inputs={"company": self.state.company, "date": datetime.now().strftime("%B %d, %Y")})
        self.state.market_research = str(result)
        return self.state.market_research

    @listen(run_market_research)
    def run_risk_assessment(self, market_research):
        risk_analyst = Agent(
            role="Risk Analyst",
            goal="Identify the biggest investment risks for {company}",
            backstory="You flag valuation, market, regulatory, and execution risks.",
            llm=llm,
            tools=[search_tool],
            verbose=False,
        )

        task = Task(
            description=(
                "Review this market research and search for current risks for {company}:\n\n"
                "{market_research}\n\n"
                "Today's date: {date}\n\n"
                "End your answer with exactly one line: Risk Level: normal OR Risk Level: high_risk."
            ),
            expected_output="A concise risk brief ending with Risk Level: normal or Risk Level: high_risk.",
            agent=risk_analyst,
            output_file="task_outputs/step_5/step5_risk_assessment.md",
        )

        crew = Crew(agents=[risk_analyst], tasks=[task], process=Process.sequential, verbose=True)
        result = crew.kickoff(inputs={"company": self.state.company, "market_research": market_research, "date": datetime.now().strftime("%B %d, %Y")})
        self.state.risk_assessment = str(result)
        self.state.risk_level = "high_risk" if "risk level: high_risk" in self.state.risk_assessment.lower() else "normal"
        return self.state.risk_assessment


sequential_flow = SequentialFinanceFlow()
sequential_flow.state.company = "NVIDIA"
_ = await sequential_flow.kickoff_async()

### Step 6: Parallel `@start` + `and_()`


In [ ]:
from time import perf_counter

from crewai.flow.flow import and_


def build_market_research_crew():
    researcher = Agent(
        role="Market Researcher",
        goal="Research recent market developments for {company}",
        backstory="You find concise, source-grounded finance updates for investors.",
        llm=llm,
        tools=[search_tool],
        verbose=False,
    )
    task = Task(
        description="Find 3 recent market developments about {company} as of {date} and explain why investors care.",
        expected_output="A concise markdown market brief with sources.",
        agent=researcher,
        output_file="task_outputs/step_6/step6_market_research.md",
    )
    return Crew(agents=[researcher], tasks=[task], process=Process.sequential, verbose=True)


def build_risk_assessment_crew():
    risk_analyst = Agent(
        role="Risk Analyst",
        goal="Identify the biggest investment risks for {company}",
        backstory="You flag valuation, market, regulatory, and execution risks.",
        llm=llm,
        tools=[search_tool],
        verbose=False,
    )
    task = Task(
        description=(
            "Find the current investment risks for {company} as of {date}. "
            "Focus on valuation, competition, regulation, demand, and execution. "
            "End your answer with exactly one line: Risk Level: normal OR Risk Level: high_risk."
        ),
        expected_output="A concise risk brief ending with Risk Level: normal or Risk Level: high_risk.",
        agent=risk_analyst,
        output_file="task_outputs/step_6/step6_risk_assessment.md",
    )
    return Crew(agents=[risk_analyst], tasks=[task], process=Process.sequential, verbose=True)


def build_portfolio_strategy_crew():
    strategist = Agent(
        role="Investment Strategist",
        goal="Turn market research and risk assessment into a portfolio recommendation for {company}",
        backstory="You convert research and risk signals into clear investment strategy.",
        llm=llm,
        verbose=False,
    )
    task = Task(
        description=(
            "Create a portfolio strategy for {company}.\n\n"
            "Market Research:\n{market_research}\n\n"
            "Risk Assessment:\n{risk_assessment}\n\n"
            "Recommend one stance: Buy, Hold, Watchlist, or Avoid. Explain the reasoning."
        ),
        expected_output="A concise strategy memo with stance, reasoning, and key watch items.",
        agent=strategist,
        output_file="task_outputs/step_6/step6_portfolio_strategy.md",
    )
    return Crew(agents=[strategist], tasks=[task], process=Process.sequential, verbose=True)


class ParallelFinanceFlow(Flow[FinanceCompanyState]):
    @start()
    def run_market_research(self):
        result = build_market_research_crew().kickoff(inputs={"company": self.state.company, "date": datetime.now().strftime("%B %d, %Y")})
        self.state.market_research = str(result)
        return self.state.market_research

    @start()
    def run_risk_assessment(self):
        result = build_risk_assessment_crew().kickoff(inputs={"company": self.state.company, "date": datetime.now().strftime("%B %d, %Y")})
        self.state.risk_assessment = str(result)
        self.state.risk_level = "high_risk" if "risk level: high_risk" in self.state.risk_assessment.lower() else "normal"
        return self.state.risk_assessment

    @listen(and_(run_market_research, run_risk_assessment))
    def run_portfolio_strategy(self):
        result = build_portfolio_strategy_crew().kickoff(
            inputs={"company": self.state.company, "market_research": self.state.market_research, "risk_assessment": self.state.risk_assessment}
        )
        self.state.portfolio_strategy = str(result)
        return self.state.portfolio_strategy


parallel_flow = ParallelFinanceFlow()
parallel_flow.state.company = "NVIDIA"
parallel_start = perf_counter()
_ = await parallel_flow.kickoff_async()
print(f"Parallel flow: {perf_counter() - parallel_start:.1f}s | Risk: {parallel_flow.state.risk_level}")

### Step 7: Conditional Routing with `@router`


In [ ]:
from crewai.flow.flow import router


class RoutedFinanceFlow(ParallelFinanceFlow):
    @router(ParallelFinanceFlow.run_portfolio_strategy)
    def route_on_risk(self):
        if self.state.risk_level == "high_risk":
            return "high_risk"
        return "approved"

    @listen("high_risk")
    def run_compliance_review(self):
        self.state.compliance_notes = (
            "Compliance reviewed the strategy because risk was marked high. "
            "Client-facing language should avoid guarantees, highlight uncertainty, and frame the stance as research commentary."
        )
        return self.state.compliance_notes

    @listen("approved")
    def approved_path(self):
        self.state.compliance_notes = "No extra compliance review required."
        return self.state.compliance_notes


routed_flow = RoutedFinanceFlow()
routed_flow.state.company = "NVIDIA"
_ = await routed_flow.kickoff_async()
print(f"Risk route: {routed_flow.state.risk_level}")

## Step 8: The Complete Finance Company Flow

This brings all the concepts together into one `FinanceCompanyFlow` class:

- Parallel `@start` — market research and risk assessment run at the same time
- `and_()` — portfolio strategy waits for both to finish
- `@router` — branches on risk level: high-risk goes through compliance, approved goes straight to the advisory
- Each branch ends by calling the client advisory crew directly — no extra merge operator needed

Call `flow.plot()` to visualize the DAG, then `flow.kickoff_async()` to run the full company.

In [ ]:
def build_client_advisory_crew():
    advisor = Agent(
        role="Client Advisor",
        goal="Write a clear investor-facing report for {company}",
        backstory="You translate internal research into plain-English client recommendations.",
        llm=llm,
        verbose=False,
    )
    task = Task(
        description=(
            "Write a client advisory report for {company}.\n\n"
            "Market Research:\n{market_research}\n\n"
            "Risk Assessment:\n{risk_assessment}\n\n"
            "Portfolio Strategy:\n{portfolio_strategy}\n\n"
            "Compliance Notes:\n{compliance_notes}\n\n"
            "Keep it concise, balanced, and useful for an investor."
        ),
        expected_output="A client-ready investor report with summary, recommendation, risks, and next steps.",
        agent=advisor,
        output_file="task_outputs/client_report.md",
    )
    return Crew(agents=[advisor], tasks=[task], process=Process.sequential, verbose=True)


def _run_advisory(state):
    return build_client_advisory_crew().kickoff(
        inputs={
            "company": state.company,
            "market_research": state.market_research,
            "risk_assessment": state.risk_assessment,
            "portfolio_strategy": state.portfolio_strategy,
            "compliance_notes": state.compliance_notes,
        }
    )


class FinanceCompanyFlow(Flow[FinanceCompanyState]):
    @start()
    def run_market_research(self):
        result = build_market_research_crew().kickoff(inputs={"company": self.state.company, "date": datetime.now().strftime("%B %d, %Y")})
        self.state.market_research = str(result)
        return self.state.market_research

    @start()
    def run_risk_assessment(self):
        result = build_risk_assessment_crew().kickoff(inputs={"company": self.state.company, "date": datetime.now().strftime("%B %d, %Y")})
        self.state.risk_assessment = str(result)
        self.state.risk_level = "high_risk" if "risk level: high_risk" in self.state.risk_assessment.lower() else "normal"
        return self.state.risk_assessment

    @listen(and_(run_market_research, run_risk_assessment))
    def run_portfolio_strategy(self):
        result = build_portfolio_strategy_crew().kickoff(
            inputs={"company": self.state.company, "market_research": self.state.market_research, "risk_assessment": self.state.risk_assessment}
        )
        self.state.portfolio_strategy = str(result)
        return self.state.portfolio_strategy

    @router(run_portfolio_strategy)
    def route_on_risk(self):
        if self.state.risk_level == "high_risk":
            return "high_risk"
        return "approved"

    @listen("high_risk")
    def run_compliance_then_advisory(self):
        self.state.compliance_notes = (
            "Compliance reviewed the strategy because risk was marked high. "
            "The final report must avoid guarantees and present the recommendation as research commentary."
        )
        self.state.client_report = str(_run_advisory(self.state))
        return self.state.client_report

    @listen("approved")
    def run_client_advisory(self):
        self.state.compliance_notes = "No extra compliance review required."
        self.state.client_report = str(_run_advisory(self.state))
        return self.state.client_report


company_flow = FinanceCompanyFlow()
company_flow.state.company = "NVIDIA"
company_start = perf_counter()
_ = await company_flow.kickoff_async()
print(f"Finance company flow: {perf_counter() - company_start:.1f}s | Risk: {company_flow.state.risk_level}")